In [ ]:
import pandas as pd
from google.colab import files

uploaded = files.upload()

file_name = list(uploaded.keys())[0]
print("Uploaded file:", file_name)

if file_name.lower().endswith(".csv"):
    df = pd.read_csv(file_name)

elif file_name.lower().endswith((".xlsx", ".xls")):
    df = pd.read_excel(file_name)

else:
    raise ValueError("Please upload a CSV or Excel file.")

print("Dataset shape:", df.shape)
display(df.head())

Saving skincare_recommendation_demo_300(1).xlsx to skincare_recommendation_demo_300(1).xlsx
Uploaded file: skincare_recommendation_demo_300(1).xlsx
Dataset shape: (300, 12)


,Type Product,Name,Brand,Composition,Price,Oily,Dry,Sensitive,Skin Type,Image,Link,Rank
0,moisturizer,Ceramide Barrier Gel Moisturizer,DermaNest,"Aqua, Glycerin, Ceramide NP, Panthenol",211.107,Low,Medium,High,Normal,https://www.google.com/search?tbm=isch&q=Ceram...,https://www.google.com/search?q=Ceramide+Barri...,1
1,facewash,Low pH Gentle Face Wash,PureRoutine,"Aqua, Decyl Glucoside, Glycerin, Centella Asia...",25.553,High,High,Medium,Dry,https://www.google.com/search?tbm=isch&q=Low+p...,https://www.google.com/search?q=Low+pH+Gentle+...,2
2,sunscreen,Mineral Calm Daily Sun Fluid SPF 50,SkinHarbor,"Aqua, Zinc Oxide, Glycerin, Caprylic/Capric Tr...",101.059,Medium,Low,Low,Oily,https://www.google.com/search?tbm=isch&q=Miner...,https://www.google.com/search?q=Mineral+Calm+D...,3
3,serum,Azelaic Support Daily Serum 30ml,CalmTheory,"Aqua, Panthenol, Allantoin, Centella Asiatica ...",85.410,Low,Medium,High,Combination,https://www.google.com/search?tbm=isch&q=Azela...,https://www.google.com/search?q=Azelaic+Suppor...,4
4,toner,Barrier Care Essence Toner 150ml,DailyKind,"Aqua, Oat Kernel Extract, Glycerin, Panthenol",240.414,High,High,Medium,Normal,https://www.google.com/search?tbm=isch&q=Barri...,https://www.google.com/search?q=Barrier+Care+E...,5


In [ ]:
# Step 2: Validate and prepare the dataset

required_cols = [
    "Type Product", "Name", "Brand", "Composition", "Price",
    "Oily", "Dry", "Sensitive", "Skin Type", "Image", "Link"
]

# Remove extra spaces from column names and text values
df.columns = df.columns.str.strip()

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

# Check required columns
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

# Check null and blank values
print("Dataset shape before validation:", df.shape)
print("\nNull values:")
print(df[required_cols].isna().sum())

# Validate labels
for col in ["Oily", "Dry", "Sensitive"]:
    print(f"\n{col} values:", df[col].unique())

print("\nSkin Type values:", df["Skin Type"].unique())
print("\nDuplicate product names:", df["Name"].duplicated().sum())

Dataset shape before validation: (300, 12)

Null values:
Type Product    0
Name            0
Brand           0
Composition     0
Price           0
Oily            0
Dry             0
Sensitive       0
Skin Type       0
Image           0
Link            0
dtype: int64

Oily values: ['Low' 'High' 'Medium']

Dry values: ['Medium' 'High' 'Low']

Sensitive values: ['High' 'Medium' 'Low']

Skin Type values: ['Normal' 'Dry' 'Oily' 'Combination']

Duplicate product names: 0


In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Features used for matching
feature_cols = ["Skin Type", "Oily", "Dry", "Sensitive"]

# One-hot encode the categorical features
encoder = OneHotEncoder(handle_unknown="ignore")

product_features = encoder.fit_transform(df[feature_cols])

print("Encoded feature matrix shape:", product_features.shape)
print("Number of products:", product_features.shape[0])
print("Number of encoded features:", product_features.shape[1])

Encoded feature matrix shape: (300, 13)
Number of products: 300
Number of encoded features: 13


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

def recommend_products(skin_type, oily, dry, sensitive, top_n=5):

    # Create user profile using the same feature columns
    user_profile = pd.DataFrame([{
        "Skin Type": skin_type.strip().title(),
        "Oily": oily.strip().title(),
        "Dry": dry.strip().title(),
        "Sensitive": sensitive.strip().title()
    }])

    # Convert user profile using the fitted encoder
    user_features = encoder.transform(user_profile)

    # Calculate cosine similarity with every product
    scores = cosine_similarity(
        user_features,
        product_features
    ).flatten()

    # Get top matching products
    results = df.copy()
    results["Similarity Score"] = scores

    results = results.sort_values(
        by="Similarity Score",
        ascending=False
    )

    return results.head(top_n)[[
        "Name",
        "Brand",
        "Type Product",
        "Price",
        "Skin Type",
        "Oily",
        "Dry",
        "Sensitive",
        "Similarity Score",
        "Image",
        "Link"
    ]]

In [ ]:
# Example CNN prediction
skin_type = "Combination"

# Example NLP predictions
oily = "High"
dry = "Low"
sensitive = "Medium"

recommendations = recommend_products(
    skin_type=skin_type,
    oily=oily,
    dry=dry,
    sensitive=sensitive,
    top_n=5
)

display(recommendations)

,Name,Brand,Type Product,Price,Skin Type,Oily,Dry,Sensitive,Similarity Score,Image,Link
281,Peptide Night Night Treatment 30ml Variant 47,SkinHarbor,treatment,79.486,Combination,High,High,Medium,0.75,https://www.google.com/search?tbm=isch&q=Pepti...,https://www.google.com/search?q=Peptide+Night+...
274,Rice Water Essence Toner 150ml Variant 46,DailyKind,toner,58.556,Combination,High,High,Medium,0.75,https://www.google.com/search?tbm=isch&q=Rice+...,https://www.google.com/search?q=Rice+Water+Ess...
30,Oat Comfort Gel Moisturizer,GlowField,moisturizer,261.753,Combination,High,High,Medium,0.75,https://www.google.com/search?tbm=isch&q=Oat+C...,https://www.google.com/search?q=Oat+Comfort+Ge...
267,Hyaluronic Hydration Daily Serum 30ml Variant 45,GlowField,serum,316.144,Combination,High,High,Medium,0.75,https://www.google.com/search?tbm=isch&q=Hyalu...,https://www.google.com/search?q=Hyaluronic+Hyd...
37,Cica Fresh Foaming Cleanser,PureRoutine,facewash,204.793,Combination,High,High,Medium,0.75,https://www.google.com/search?tbm=isch&q=Cica+...,https://www.google.com/search?q=Cica+Fresh+Foa...


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Handle any missing ingredient text safely
df["Composition"] = df["Composition"].fillna("").astype(str)

# Convert product ingredients into TF-IDF vectors
ingredient_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)

ingredient_features = ingredient_vectorizer.fit_transform(
    df["Composition"]
)

print("Ingredient matrix shape:", ingredient_features.shape)
print("Products:", len(df))
print("Ingredient features:", ingredient_features.shape[1])

Ingredient matrix shape: (300, 172)
Products: 300
Ingredient features: 172


In [ ]:
def recommend_products(
    skin_type, oily, dry, sensitive, top_n=5
):
    user_profile = pd.DataFrame([{
        "Skin Type": skin_type.strip().title(),
        "Oily": oily.strip().title(),
        "Dry": dry.strip().title(),
        "Sensitive": sensitive.strip().title()
    }])

    # Skin attribute similarity
    user_features = encoder.transform(user_profile)

    skin_scores = cosine_similarity(
        user_features, product_features
    ).flatten()

    # Ingredient similarity:
    # This basic version uses the product's category as a proxy for
    # the user's desired product type, not a medical ingredient profile.
    category_scores = np.zeros(len(df))

    # Combine the two scores
    final_scores = 0.7 * skin_scores + 0.3 * category_scores

    results = df.copy()
    results["Similarity Score"] = final_scores

    return results.sort_values(
        "Similarity Score", ascending=False
    )[[
        "Name", "Brand", "Type Product", "Price",
        "Skin Type", "Oily", "Dry", "Sensitive",
        "Composition", "Similarity Score", "Image", "Link"
    ]].head(top_n)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

def recommend_products(
    skin_type, oily, dry, sensitive, top_n=5
):
    # Validate user input
    valid_levels = {"Low", "Medium", "High"}
    valid_skin_types = {"Normal", "Dry", "Oily", "Combination"}

    skin_type = skin_type.strip().title()
    oily = oily.strip().title()
    dry = dry.strip().title()
    sensitive = sensitive.strip().title()

    if skin_type not in valid_skin_types:
        raise ValueError(f"Skin Type must be one of {valid_skin_types}")

    if not {oily, dry, sensitive}.issubset(valid_levels):
        raise ValueError("Oily, Dry, and Sensitive must be Low, Medium, or High")

    # Encode user attributes with the already-fitted encoder
    user_profile = pd.DataFrame([{
        "Skin Type": skin_type,
        "Oily": oily,
        "Dry": dry,
        "Sensitive": sensitive
    }])

    user_features = encoder.transform(user_profile)

    # Compare with every product
    scores = cosine_similarity(
        user_features,
        product_features
    ).ravel()

    results = df.copy()
    results["Similarity Score"] = scores

    # Return highest-scoring products
    results = results.sort_values(
        "Similarity Score", ascending=False
    ).head(top_n)

    display_cols = [
        "Name", "Brand", "Type Product", "Price",
        "Skin Type", "Oily", "Dry", "Sensitive",
        "Composition", "Similarity Score", "Image", "Link"
    ]

    return results[display_cols].reset_index(drop=True)

In [ ]:
recommendations = recommend_products(
    skin_type="Combination",  # Example CNN output
    oily="High",              # Example NLP output
    dry="Low",
    sensitive="Medium",
    top_n=5
)

display(recommendations)

,Name,Brand,Type Product,Price,Skin Type,Oily,Dry,Sensitive,Composition,Similarity Score,Image,Link
0,Peptide Night Night Treatment 30ml Variant 47,SkinHarbor,treatment,79.486,Combination,High,High,Medium,"Aqua, Centella Asiatica Extract, Allantoin, Gl...",0.75,https://www.google.com/search?tbm=isch&q=Pepti...,https://www.google.com/search?q=Peptide+Night+...
1,Rice Water Essence Toner 150ml Variant 46,DailyKind,toner,58.556,Combination,High,High,Medium,"Aqua, Oat Kernel Extract, Glycerin, Panthenol",0.75,https://www.google.com/search?tbm=isch&q=Rice+...,https://www.google.com/search?q=Rice+Water+Ess...
2,Oat Comfort Gel Moisturizer,GlowField,moisturizer,261.753,Combination,High,High,Medium,"Aqua, Squalane, Niacinamide, Sodium Hyaluronate",0.75,https://www.google.com/search?tbm=isch&q=Oat+C...,https://www.google.com/search?q=Oat+Comfort+Ge...
3,Hyaluronic Hydration Daily Serum 30ml Variant 45,GlowField,serum,316.144,Combination,High,High,Medium,"Aqua, Peptides, Glycerin, Sodium Hyaluronate",0.75,https://www.google.com/search?tbm=isch&q=Hyalu...,https://www.google.com/search?q=Hyaluronic+Hyd...
4,Cica Fresh Foaming Cleanser,PureRoutine,facewash,204.793,Combination,High,High,Medium,"Aqua, Decyl Glucoside, Glycerin, Centella Asia...",0.75,https://www.google.com/search?tbm=isch&q=Cica+...,https://www.google.com/search?q=Cica+Fresh+Foa...
